In [1]:
import numpy as np
import pandas as pd
from os import listdir
import re
from os.path import isfile, join
import os

# Formatting the intensities

The aim of this section is to aggregate all the data into a unique csv file. I add the ROI as a column

In [2]:
path0 = '../IMC_SegmentationResults/'
path = path0+'all_data/'
directory = 'intensities/'
file_names = [f for f in listdir(path+directory) if isfile(join(path+directory, f))]
stripped_names = [re.sub(r'\.[^.]+$', '', name) for name in file_names]

In [3]:
stripped_names

['Leap002_002',
 'Leap002_003',
 'Leap002_006',
 'Leap001_008',
 'Leap001_009',
 'Leap002_004',
 'Leap002_005',
 'Leap001_010',
 'Leap002_007']

In [4]:
response_data = pd.read_excel('input_files/LEAP code response data 020523.xlsx')
#response_data['LEAP ID']
response_data

,LEAP ID,Response,COMMENTS,note if frozen (otherwise FFPE),Extreme non-responder (death within 2 years?)
0,LEAP001/2,Non-Responder,1=core 2= resection,NaN,NaN
1,LEAP003/4,Non-Responder,3=core 4=resection,NaN,NaN
2,LEAP005/6,Non-Responder,5=core 6=resection,NaN,NaN
3,LEAP007,pCR,core,NaN,NaN
4,LEAP008,pCR,core,NaN,Y
...,...,...,...,...,...
64,LEAP090,pCR,core,NaN,NaN
65,LEAP091,pCR,core,NaN,NaN
66,LEAP092,pCR,core,NaN,NaN
67,LEAP093/94,Non-Responder,93= core 94 = resection,NaN,Y


In [5]:
response_data['LEAP ID'].str

For every intensity file, I associate the appropriate ROI

In [6]:
images = pd.read_csv(path+'images.csv',index_col=0)
images['filename'] = images.image
images['ROI'] = images.acquisition_description.str.extract(r'(ROI_\d+)')#extract ROI from string
images.image = images.image.str.replace( r'\.[^.]+$','',regex = True)#strip  file extension
images =images[~images.duplicated(subset = ['image','ROI'])]
common_set = set(stripped_names)&set(images.image)
images = images[images.image.isin(common_set)][['image','ROI']]
images.set_index('image',inplace=True)
images


,ROI
image,
Leap001_008,ROI_001
Leap001_009,ROI_002
Leap001_010,ROI_003
Leap002_003,ROI_007
Leap002_004,ROI_005
Leap002_005,ROI_004
Leap002_006,ROI_003
Leap002_007,ROI_002


I also add a column called ``source_file`` just to check in the future for batch effects. AnnoSpat doesn't care as long it is after the ROI.

In [7]:
merged_intensities = pd.DataFrame()
for file in common_set:
    df = pd.read_csv(path+directory+file+'.csv')
    df = df[df.columns[df.columns.str.find('-')>0]]#select only columns related to antibody and filter out empty channels
    df.columns = df.columns.str.replace(r'^.*?-','',regex = True)#remove the metals

    df['ROI'] = images.loc[file].values[0]
    df['source_file'] = file
    merged_intensities = pd.concat([merged_intensities,df],ignore_index=True)
print('shape of the dataframe:',merged_intensities.shape)
merged_intensities.head()

shape of the dataframe: (58031, 39)


,CD38,EGFR,p53,CD14,Tbet,CD16,CD163,Pan-keratin,CD11b,PD-L1,...,CD27,PD-L2,CD45RO,Alpha-SMA,Vimentin,CD31,DNA1,DNA2,ROI,source_file
0,0.381761,1.442329,0.230218,0.333333,0.000000,0.166667,0.000000,0.000000,0.982985,0.234284,...,0.000000,0.258033,0.242131,0.570516,1.573388,0.166667,1.412595,1.180210,ROI_005,Leap002_004
1,0.231574,0.395118,0.051282,0.632789,0.260964,0.472907,0.094213,0.402510,0.722732,0.137797,...,0.223786,0.375983,0.555141,0.617912,1.434125,0.186116,5.162425,7.483028,ROI_005,Leap002_004
2,0.050000,0.702630,0.050000,0.250000,0.286646,0.441755,0.050000,0.000000,1.878291,0.150000,...,0.150000,0.272937,0.715553,1.052203,1.648499,0.068163,5.466761,7.918406,ROI_005,Leap002_004
3,0.353633,0.890232,0.217234,0.504322,0.179026,0.211257,0.082921,0.278361,0.811403,0.198278,...,0.234712,0.210134,1.485286,0.569825,2.610184,0.105057,5.273431,8.698304,ROI_005,Leap002_004
4,0.356444,0.392755,0.494036,0.519169,0.425285,0.270693,0.148164,0.273558,2.116173,0.222280,...,0.085714,0.172301,1.720458,0.610585,0.707895,0.096972,3.708644,7.927870,ROI_005,Leap002_004


save the file

In [8]:
path2 = './processed_files'
isExist = os.path.exists(path)
if not isExist:

   # Create a new directory because it does not exist
   os.makedirs(path)
merged_intensities.to_csv(path2+'/all_data_intensities.csv')

Concatenate together the region files which contains the spatial information for every cell

In [9]:
merged_regions = pd.DataFrame()
for file in common_set:
    df = pd.read_csv(path+'/regionprops/'+file+'.csv')
    df['source_file'] = file

    merged_regions = pd.concat([merged_regions,df.iloc[:,1:]],ignore_index=True)#remove the object column
    
merged_regions.to_csv('./processed_files/all_data_regions.csv')

# Creating the signature
I format the signature file to work with AnnoSpat

In [14]:
#signature = pd.read_csv('../IMC_SegmentationResults/IMC_SegmentationResults/IMCCelltypeResults/cell_type_matrix.csv').set_index('Marker')
signature = pd.read_csv(path0+'/IMCCelltypeResults/cell_type_matrix.csv').set_index('Marker')
signature.head()
signature.drop(labels=['Proliferative cells?','Other cells'],axis='columns',inplace=True)# drop the proliferative cell class
signature.head()

,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,NK and CD8 Tcell,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,p53+ cells?,Neutrophil&monocyte
Marker,,,,,,,,,,,,,,,,,,,
CD38,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EGFR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
p53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
CD14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,-1.0,-1.0,NaN,NaN,NaN,-1.0
Tbet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
new = {}
for column in signature.columns:
    protein_2_add = signature.index[~signature[column].isna()]#series of protein to use as markers
    a =signature[column][protein_2_add]
    new[column] = list(protein_2_add.where(a>0,protein_2_add.astype(str)+'-'))
signature =pd.DataFrame.from_dict(new,orient = 'index').T

In [17]:
signature.to_csv('processed_files/signature.csv')    

In [16]:
signature

,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,NK and CD8 Tcell,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,p53+ cells?,Neutrophil&monocyte
0,Pan-keratin,CD163,CD163-,CD38,CD38,CD38,CD38,CD38,CD38,CD38,CD38,CD14,CD14,CD14-,CD14-,Pan-keratin-,Pan-keratin,p53,CD14-
1,E-Cadherin,Pan-keratin-,Pan-keratin-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,Pan-keratin-,CD16-,CD16,CD16,CD16-,CD45-,E-Cadherin,Ki-67,CD16
2,Beta-Catenin,CD68,CD68,CD45,CD45,CD45,CD45,CD45,CD45,CD45,CD107a,Pan-keratin-,Pan-keratin-,Pan-keratin-,Pan-keratin-,Alpha-SMA,Beta-Catenin,None,CD11b
3,None,None,None,CD44-,CD44,CD44,CD44-,CD44,FOXP3-,FOXP3-,Granzyme-B,CD68-,CD68-,CD68-,CD68-,Vimentin,B7-H4,None,CD45
4,None,None,None,FOXP3-,FOXP3-,FOXP3,FOXP3-,FOXP3-,CD20,CD20,None,CD20-,CD20-,CD20-,CD20-,CD31-,None,None,None
5,None,None,None,CD4,CD4,CD4,CD4-,CD4-,CD3-,CD3-,None,CD3-,CD3-,CD3-,CD3-,None,None,None,None
6,None,None,None,CD20-,CD20-,CD20-,CD20-,CD20-,CD27-,CD27,None,None,None,None,CD31,None,None,None,None
7,None,None,None,CD8a-,CD8a-,CD8a-,CD8a,CD8a,None,None,None,None,None,None,None,None,None,None,None
8,None,None,None,CD3,CD3,CD3,CD3,CD3,None,None,None,None,None,None,None,None,None,None,None
9,None,None,None,CD27,CD27,CD27,CD27-,CD27,None,None,None,None,None,None,None,None,None,None,None


COde below run AnnoSpat to perform cell type annotation

In [18]:
!python AnnoSpat_main/AnnoSpat/run.py -i processed_files/all_data_intensities.csv  -m processed_files/signature.csv  -o ./output -f CD38 -l DNA2 -r ROI -a '[99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5,99.5]'


=====Function chosen: ANNOTATION=====

Starting cell type annotation.....

-----Estimating cell types using Semi supervised clustering-----

-------------------------Done-------------------------

Time taken: 0.0227058211962382seconds

Estimating cell types...
-----Estimating cell types using ELM-----

-------------------------Done-------------------------

Time taken: 0.0371102253595988seconds

Saving Annotations in: /home/giuseppe/Downloads/Annospat_attempt/output/trte_labels_numericLabels_ELM_IMC_T1D_AnnoSpat.csv
-------------------------Done-------------------------



In [21]:
pd.__version__

'2.0.3'